In [3]:
import pandas as pd
import numpy as np
import warnings
import geopandas as gpd

warnings.filterwarnings('ignore')

In [4]:
url1 = "https://raw.githubusercontent.com/statzenthusiast921/wildfires/refs/heads/main/data/fire_df_wa.csv"
url2 = "https://raw.githubusercontent.com/statzenthusiast921/wildfires/refs/heads/main/data/fire_df_or.csv"
url3 = "https://raw.githubusercontent.com/statzenthusiast921/wildfires/refs/heads/main/data/fire_df_ca.csv"

df_wa = pd.read_csv(url1)
df_or = pd.read_csv(url2)
df_ca = pd.read_csv(url3, dtype={16: str, 18: str})

In [5]:
df = pd.concat([df_wa, df_or, df_ca], ignore_index=True)
df.shape

(284151, 19)

In [6]:
df['EndDate'] = pd.to_datetime(df['CONT_DATE'], unit='D', origin='julian')
df['StartDate'] = pd.to_datetime(df['DISCOVERY_DATE'], unit='D', origin='julian')
df['FireLengthDays'] = df['CONT_DATE'] - df['DISCOVERY_DATE']

In [7]:
df.head()

,NWCG_REPORTING_UNIT_NAME,FIRE_NAME,FIRE_YEAR,DISCOVERY_DATE,DISCOVERY_DOY,DISCOVERY_TIME,STAT_CAUSE_CODE,STAT_CAUSE_DESCR,CONT_DATE,CONT_DOY,...,FIRE_SIZE_CLASS,LATITUDE,LONGITUDE,STATE,COUNTY,FIPS_CODE,FIPS_NAME,EndDate,StartDate,FireLengthDays
0,Umatilla National Forest,UPPER JIM,2005,2453540.5,170,1430.0,1.0,Lightning,2453541.5,171.0,...,B,46.220833,-117.785000,WA,27,27.0,Grays Harbor,2005-06-20,2005-06-19,1.0
1,Umatilla National Forest,SKYLINE,2005,2453567.5,197,1354.0,4.0,Campfire,2453567.5,197.0,...,A,46.080556,-117.890833,WA,13,13.0,Columbia,2005-07-16,2005-07-16,0.0
2,Columbia River Gorge National Scenic Area,MP 80,2005,2453516.5,146,950.0,6.0,Railroad,2453516.5,146.0,...,A,45.665000,-121.203611,WA,39,39.0,Klickitat,2005-05-26,2005-05-26,0.0
3,Idaho Panhandle National Forest,BUTCH CREEK,2005,2453585.5,215,1236.0,9.0,Miscellaneous,2453585.5,215.0,...,A,48.356667,-117.060833,WA,51,51.0,Pend Oreille,2005-08-03,2005-08-03,0.0
4,Columbia River Gorge National Scenic Area,LOCKE LAKE,2005,2453555.5,185,2002.0,6.0,Railroad,2453555.5,185.0,...,A,45.699444,-121.413611,WA,39,39.0,Klickitat,2005-07-04,2005-07-04,0.0


In [8]:
prefix_map = {"WA": "53", "CA": "06", "OR": "41"}

def make_fips(row):
    if pd.isna(row["FIPS_CODE"]):
        return np.nan
    
    county_code = str(int(row["FIPS_CODE"])).zfill(3)
    state_prefix = prefix_map.get(row["STATE"], "")
    
    return state_prefix + county_code if state_prefix else np.nan

df["FIPS"] = df.apply(make_fips, axis=1)

In [9]:
df["FIPS_length"] = df["FIPS"].astype(str).str.len()
df["FIPS_length"].value_counts()

FIPS_length
3    164516
5    119635
Name: count, dtype: int64

In [10]:
df_good = df[df['FIPS_length']==5]
df_bad = df[df['FIPS_length']==3]

In [12]:
from shapely.geometry import Point
counties = gpd.read_file("/Users/jonzimmerman/Desktop/Data Projects/Wildfires/data/cb_2018_us_county_500k/cb_2018_us_county_500k.shp")
counties = counties[counties['STATEFP'].isin(['06','41','53'])]  # CA=06, OR=41, WA=53
counties = counties[['STATEFP', 'COUNTYFP', 'GEOID', 'geometry']]  # GEOID is full FIPS

# 2. Convert your df to a GeoDataFrame
gdf = gpd.GeoDataFrame(
    df_bad,
    geometry=gpd.points_from_xy(df_bad["LONGITUDE"], df_bad["LATITUDE"]),
    crs="EPSG:4326"
)

# 3. Spatial join
gdf = gpd.sjoin(gdf, counties, how="left", predicate="within")

# 4. Fill missing FIPS with GEOID from join
gdf["FIPS"] = gdf["FIPS"].fillna(gdf["GEOID"])

# 5. Drop geometry columns if you want back a regular DataFrame
df_bad = pd.DataFrame(gdf.drop(columns=['geometry', 'index_right', 'STATEFP', 'COUNTYFP', 'GEOID']))


In [18]:
df_all = pd.concat([df_bad, df_good], ignore_index=True)

In [20]:
#Recalculate
df_all["FIPS_length"] = df_all["FIPS"].astype(str).str.len()
df_all["FIPS_length"].value_counts()

FIPS_length
5    283906
3       245
Name: count, dtype: int64

In [27]:
df_all = df_all[df_all['FIPS_length']==5]

In [29]:
county_counts = df_all.groupby("STATE")["FIPS"].nunique().reset_index()
county_counts

,STATE,FIPS
0,CA,60
1,OR,41
2,WA,43
